# 03 · Análisis exploratorio con DataFrames

**Objetivos de aprendizaje**

- Seleccionar, filtrar, ordenar y crear columnas con la API de DataFrames.
- Calcular estadísticas descriptivas y percentiles de forma distribuida.
- Analizar frecuencias de variables categóricas.
- Construir histogramas, matrices de correlación y series de tiempo.
- Trasladar a pandas únicamente resultados agregados para graficar.

| Aspecto | Detalle |
|---|---|
| Duración estimada | 40 minutos |
| Requisito previo | Cuaderno 01 |
| Entrada | Dataset activo (versión original) |
| Salida | Hallazgos del análisis exploratorio |

### Cómo utilizar este cuaderno

1. Las celdas se ejecutan **en orden**, de arriba hacia abajo, con `Shift + Enter`. Cada celda puede depender de variables creadas en celdas anteriores.
2. Las celdas que comienzan con `# >>> CELDA EDITABLE` son las únicas que deben modificarse para adaptar el análisis a otro dataset. El resto del código se ajusta automáticamente a las columnas detectadas.
3. Cada sección sigue la misma estructura: **fundamento** (qué es y por qué se hace), **código**, **interpretación** del resultado y **adaptación** al dataset propio.
4. Ante un error `NameError` (variable no definida), lo habitual es que no se haya ejecutado una celda anterior. La solución es *Kernel > Restart Kernel and Run All Cells*.
5. Mientras la sesión de Spark está activa, la Spark UI (normalmente <http://localhost:4040>) muestra los *jobs*, *stages* y tareas ejecutados.

La guía completa de uso de los cuadernos se encuentra en `docs/08-uso-de-notebooks.md`.

## Preparación del entorno

La celda siguiente realiza cuatro tareas:

1. **Ubica la carpeta del taller** con `ubicar_taller()`, que busca `scripts/taller_spark.py` a partir de la carpeta actual. Esto evita errores de ruta del tipo `Input path does not exist: file:/data/...`, que aparecen cuando el cuaderno se abre desde una ubicación distinta, como ocurre en Google Colab, donde la carpeta de trabajo es `/content`. Después fija la carpeta `notebooks/` como carpeta de trabajo.
2. **Solo en Google Colab**: busca el taller en cualquier carpeta de `/content` y en Google Drive; si no lo encuentra, lo descarga con `git clone` desde la dirección indicada en `REPO_URL` (por eso, en Colab, debe reemplazarse `<usuario>` por el usuario de GitHub donde está publicado el taller). También instala las dependencias que falten en Colab.
3. Agrega la carpeta `scripts/` a la ruta de búsqueda de Python, para poder importar el módulo `taller_spark.py`, que contiene las funciones de apoyo del taller.
4. Importa `functions` de PySpark con el alias `F`, convención habitual en la documentación oficial (The Apache Software Foundation [ASF], 2026b).
Finalmente, crea la `SparkSession` mediante `iniciar_spark()`. Esta función verifica que Java sea la versión 17 o 21, utiliza el mismo intérprete de Python en el *driver* y en los *executors* y se conecta al clúster de Docker cuando existe la variable `SPARK_MASTER`; en cualquier otro caso trabaja en modo `local[*]`.

In [ ]:
# Preparación del entorno: se ejecuta siempre en primer lugar
import glob, os, subprocess, sys
from pathlib import Path

# >>> CELDA EDITABLE (solo Google Colab): dirección del repositorio del taller en GitHub
REPO_URL = "https://github.com/<usuario>/taller-apache-spark.git"

def en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def ubicar_taller():
    """Devuelve la carpeta raíz del taller: la que contiene scripts/taller_spark.py."""
    actual = Path.cwd().resolve()
    candidatos = [actual, *actual.parents]
    for patron in ("/content/*/scripts/taller_spark.py",                  # clonado en Colab
                   "/content/drive/MyDrive/*/scripts/taller_spark.py",    # copiado en Google Drive
                   "/content/drive/MyDrive/*/*/scripts/taller_spark.py"):
        candidatos += [Path(p).parent.parent for p in sorted(glob.glob(patron))]
    for base in candidatos:
        if (base / "scripts" / "taller_spark.py").exists():
            return base
    if en_colab():
        if "<usuario>" in REPO_URL:
            raise RuntimeError("En Google Colab debe escribirse en REPO_URL la dirección del repositorio "
                               "del taller (reemplazar <usuario>) y volver a ejecutar esta celda.")
        destino = Path("/content") / Path(REPO_URL.rstrip("/")).stem        # nombre del repositorio
        subprocess.run(["git", "clone", "-q", REPO_URL, str(destino)], check=True)
        return destino
    raise FileNotFoundError("No se encontró la carpeta del taller (scripts/taller_spark.py). "
                            "Abrir el cuaderno desde la carpeta notebooks/ del repositorio.")

RAIZ_TALLER = ubicar_taller()
os.chdir(RAIZ_TALLER / "notebooks")              # rutas iguales en cualquier entorno (local, Docker, Colab)
sys.path.insert(0, str(RAIZ_TALLER / "scripts"))

if en_colab():                                    # en Colab se instalan las dependencias que falten
    import importlib.util
    faltan = [m for m in ("openpyxl",) if importlib.util.find_spec(m) is None]
    try:
        import pyspark
        if int(pyspark.__version__.split(".")[0]) < 4:
            faltan.append("pyspark==4.3.0")
    except ImportError:
        faltan.append("pyspark==4.3.0")
    if faltan:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Carpeta del taller:", RAIZ_TALLER)

from taller_spark import *                       # iniciar_spark, cargar_desde_config, perfil, ...
from pyspark.sql import functions as F           # funciones de columnas de Spark

spark = iniciar_spark("03_Exploracion")

## Carga del dataset activo

El dataset activo es el que se configuró en el cuaderno `01_configuracion_dataset.ipynb` y quedó guardado en `data/dataset_config.json`. Si no se ha configurado ninguno, se utiliza el dataset de ejemplo (`data/ejemplo/ventas_tienda.csv`).

La función `clasificar_columnas()` asigna a cada columna un rol, que los cuadernos usan para decidir qué análisis aplicar:

| Variable | Rol | Criterio |
|---|---|---|
| `NUM` | numéricas | tipos numéricos que no son identificadores |
| `CAT` | categóricas | texto o booleanos con pocos valores distintos (hasta `max_categorias`) |
| `FEC` | fechas | tipos `date` o `timestamp` |
| `TXT` | texto | texto con muchos valores distintos (comentarios, descripciones) |
| — | identificadores | columnas excluidas en la configuración o con valores casi únicos |

Con un dataset propio, conviene revisar esta clasificación. Si una columna quedó en un rol inadecuado, puede corregirse en la configuración (`columnas_excluir`, `max_categorias`) o directamente en las listas, por ejemplo `CAT.remove("columna")`.

In [ ]:
# Carga del dataset activo (definido y guardado en el cuaderno 01)
df, cfg = cargar_desde_config(spark, version="original")

# Clasificación automática de columnas por rol analítico
roles = clasificar_columnas(df, excluir=cfg["columnas_excluir"], max_categorias=cfg["max_categorias"])
NUM, CAT, FEC, TXT = roles["numericas"], roles["categoricas"], roles["fechas"], roles["texto"]
objetivo = columna_objetivo(cfg, df)
resumen_roles(roles)
print("Columna objetivo:", objetivo)

## 1. Fundamento: análisis exploratorio de datos

El análisis exploratorio (*exploratory data analysis*) busca conocer la estructura, la calidad y los patrones de un conjunto de datos antes de formular conclusiones o modelos (Tukey, 1977). En Spark, cada cálculo se distribuye entre las particiones, y solo los resultados resumidos llegan a la memoria del equipo para graficarlos.

## 2. Operaciones básicas con DataFrames

| Operación | Método | Tipo |
|---|---|---|
| Elegir columnas | `select()` | transformación |
| Filtrar filas | `filter()` o `where()` | transformación |
| Crear o reemplazar una columna | `withColumn()` | transformación |
| Ordenar | `orderBy()` | transformación amplia |
| Mostrar filas | `show()` | acción |
| Contar | `count()` | acción |

La celda siguiente aplica estas operaciones sobre columnas detectadas automáticamente: filtra los registros cuyo valor numérico supera la mediana y los ordena de mayor a menor.

In [ ]:
if NUM:
    n1 = objetivo if objetivo in NUM else NUM[0]
    mediana = df.approxQuantile(n1, [0.5], 0.01)[0]
    columnas = ([CAT[0]] if CAT else []) + ([FEC[0]] if FEC else []) + [n1]
    (df.select(*columnas)
       .filter(F.col(n1) > mediana)
       .orderBy(F.desc(n1))
       .show(5))
    print(f"Filtro aplicado: {n1} > {mediana:,.2f} (mediana)")

`withColumn` permite derivar nuevas variables. Como ejemplo genérico, se crea la columna `<numérica>_miles`, que expresa el valor en miles.

In [ ]:
if NUM:
    df.withColumn(f"{n1}_miles", F.round(F.col(n1) / 1000, 2)).select(n1, f"{n1}_miles").show(5)

## 3. Perfil de calidad

El perfil resume cada columna: tipo, rol, nulos, valores distintos aproximados, mínimo y máximo. Es el punto de partida para decidir la limpieza (cuaderno 04).

In [ ]:
tabla_perfil = perfil(df, roles)
tabla_perfil

**Interpretación.** Deben anotarse: (a) columnas con un porcentaje de nulos alto; (b) mínimos o máximos imposibles (por ejemplo, cantidades negativas o edades superiores a 120); (c) columnas categóricas con más valores distintos de los esperados, señal de inconsistencias de escritura.

## 4. Estadísticas descriptivas de las variables numéricas

`describe()` calcula conteo, media, desviación estándar, mínimo y máximo. `approxQuantile()` calcula percentiles con un error relativo controlado (aquí, 1 %) sin ordenar todos los datos, lo que lo hace adecuado para grandes volúmenes (ASF, 2026b).

In [ ]:
if NUM:
    df.select(NUM).describe().show(truncate=False)

In [ ]:
import pandas as pd

if NUM:
    probabilidades = [0.0, 0.25, 0.5, 0.75, 1.0]
    cuantiles = df.approxQuantile(NUM, probabilidades, 0.01)
    display(pd.DataFrame(cuantiles, index=NUM, columns=["mín", "Q1", "mediana", "Q3", "máx"]).round(2))

**Interpretación.** Una media muy distinta de la mediana indica asimetría o valores atípicos. La diferencia entre Q3 y Q1 es el rango intercuartílico (RIC), que se usa en el cuaderno 04 para detectar atípicos (Tukey, 1977).

## 5. Frecuencias de variables categóricas

Para cada columna categórica (hasta seis) se muestran los valores más frecuentes y su porcentaje sobre el total.

In [ ]:
total = df.count()
for c in CAT[:6]:
    print(f"\n=== {c} ===")
    (df.groupBy(c).count()
       .withColumn("porcentaje", F.round(100 * F.col("count") / total, 2))
       .orderBy(F.desc("count"))
       .show(8, truncate=False))

## 6. Histogramas

Un histograma muestra la distribución de una variable numérica. El cálculo de los intervalos (*bins*) y sus frecuencias se realiza en Spark con `rdd.histogram()`; a la memoria local solo llegan veinte números por variable, de modo que la técnica funciona con cualquier volumen de datos.

In [ ]:
import matplotlib.pyplot as plt

graficar = NUM[:4]
if graficar:
    fig, ejes = plt.subplots(1, len(graficar), figsize=(4.5 * len(graficar), 3.5), squeeze=False)
    for eje, c in zip(ejes[0], graficar):
        limites, frecuencias = df.select(c).dropna().rdd.flatMap(lambda fila: fila).histogram(20)
        eje.bar(limites[:-1], frecuencias, width=[b - a for a, b in zip(limites[:-1], limites[1:])], align="edge")
        eje.set_title(c); eje.set_ylabel("frecuencia")
    plt.tight_layout(); plt.show()

## 7. Relación entre variables

### 7.1 Variable numérica según una categórica

La agregación por grupos responde preguntas como "¿qué categoría concentra los mayores valores?".

In [ ]:
if CAT and NUM:
    (df.groupBy(CAT[0])
       .agg(F.count("*").alias("registros"),
            F.avg(n1).cast("decimal(22,2)").alias(f"promedio_{n1}"),
            F.sum(n1).cast("decimal(22,2)").alias(f"suma_{n1}"))
       .orderBy(F.desc(f"suma_{n1}"))
       .show(10, truncate=False))

### 7.2 Matriz de correlación

El coeficiente de correlación de Pearson mide la relación lineal entre dos variables numéricas (de −1 a 1). MLlib lo calcula de forma distribuida a partir de un vector que reúne las columnas (Meng et al., 2016).

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

if len(NUM) >= 2:
    vec = VectorAssembler(inputCols=NUM, outputCol="v", handleInvalid="skip").transform(df.select(NUM))
    matriz = Correlation.corr(vec, "v").head()[0].toArray()
    corr = pd.DataFrame(matriz, index=NUM, columns=NUM).round(2)

    plt.figure(figsize=(1 + 0.8 * len(NUM), 0.8 * len(NUM)))
    plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    plt.xticks(range(len(NUM)), NUM, rotation=45, ha="right"); plt.yticks(range(len(NUM)), NUM)
    for i in range(len(NUM)):
        for j in range(len(NUM)):
            plt.text(j, i, f"{corr.iat[i, j]:.2f}", ha="center", va="center", fontsize=8)
    plt.colorbar(label="correlación"); plt.title("Matriz de correlación"); plt.tight_layout(); plt.show()
else:
    print("Se requieren al menos dos columnas numéricas.")

**Interpretación.** Valores cercanos a 1 o −1 indican una relación lineal fuerte. Una correlación alta entre una variable explicativa y la variable objetivo puede ser útil para el modelo (cuaderno 06), pero también puede revelar que una variable se calcula a partir de la otra; en el ejemplo, `total` depende de `cantidad` y `precio_unitario`.

## 8. Análisis temporal

Si existe una columna de fecha, se agrupan los registros por mes con `date_trunc` y se grafica la evolución del número de registros y de la variable numérica principal.

In [ ]:
col_fecha = normalizar_nombre(cfg["columna_fecha"]) if cfg.get("columna_fecha") else None
col_fecha = col_fecha if col_fecha in FEC else (FEC[0] if FEC else None)

if col_fecha:
    agregados = [F.count("*").alias("registros")] + ([F.sum(n1).alias(n1)] if NUM else [])
    serie = (df.groupBy(F.date_trunc("month", F.col(col_fecha)).alias("periodo"))
               .agg(*agregados).orderBy("periodo").toPandas())      # resultado pequeño: un valor por mes
    fig, ejes = plt.subplots(1, len(agregados), figsize=(6 * len(agregados), 3.5), squeeze=False)
    for eje, medida in zip(ejes[0], serie.columns[1:]):
        eje.plot(serie["periodo"], serie[medida], marker="o")
        eje.set_title(f"{medida} por mes"); eje.tick_params(axis="x", rotation=45)
    plt.tight_layout(); plt.show()
else:
    print("El dataset no tiene columnas de fecha.")

**Interpretación con el dataset de ejemplo.** Diciembre muestra un aumento marcado de las ventas, coherente con la temporada de fin de año que se incluyó en los datos sintéticos.

## 9. Registro de hallazgos

Antes de continuar, se recomienda resumir en la celda Markdown siguiente los principales hallazgos: problemas de calidad, variables más relevantes, patrones temporales y preguntas que surgen para el análisis SQL y el modelo.

**Hallazgos del análisis exploratorio** (doble clic para editar):

1. ...
2. ...
3. ...

## Ejercicios propuestos

1. Para la segunda columna categórica (`CAT[1]`), obtener el promedio de cada variable numérica por categoría.
2. Calcular la proporción de nulos de cada columna y ordenar el resultado de mayor a menor.
3. Construir el histograma de la variable objetivo con 40 intervalos y describir su forma.
4. Si existe fecha, comparar el promedio de la variable principal por día de la semana (`F.dayofweek`).

In [ ]:
spark.stop()

## Referencias

The Apache Software Foundation. (2026b). *Spark SQL, DataFrames and Datasets guide* (Versión 4.3.0) [Documentación técnica]. https://spark.apache.org/docs/latest/sql-programming-guide.html

Meng, X., Bradley, J., Yavuz, B., Sparks, E., Venkataraman, S., Liu, D., Freeman, J., Tsai, D. B., Amde, M., Owen, S., Xin, D., Xin, R., Franklin, M. J., Zadeh, R., Zaharia, M., & Talwalkar, A. (2016). MLlib: Machine learning in Apache Spark. *Journal of Machine Learning Research, 17*(34), 1–7. https://jmlr.org/papers/v17/15-237.html

Tukey, J. W. (1977). *Exploratory data analysis*. Addison-Wesley.